# 02 · Volatility clustering & estimator efficiency

Three textbook facts about volatility, each surfaced by a sabia **volatility** / **distribution** feature:

1. **Clustering** — large moves bunch together (Mandelbrot 1963). Returns are ~uncorrelated, but *absolute* returns are strongly autocorrelated.
2. **Estimator efficiency** — range/OHLC estimators (Parkinson, Garman-Klass, Rogers-Satchell, Yang-Zhang) use intrabar information, so they track the same true volatility with less sampling noise than close-to-close.
3. **Leverage effect** — downside shocks raise volatility more than upside ones (Black 1976), so downside vol runs hotter.

The data is a **GJR-GARCH(1,1)** simulation (`_simulate.garch_ohlcv`) — clustering and leverage are baked into the process so the features can recover them. Requires `pip install plotly nbformat jupyterlab`.

In [1]:
import sys, pathlib
HERE = pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent))

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

import sabia
import _simulate as sim
import _stats as st

SCHEMA = sim.default_schema()
ANN = np.sqrt(252.0)  # daily return-std -> annualized

# A GJR-GARCH(1,1) close path: variance clusters and reacts more to downside shocks (leverage).
frame = sim.garch_ohlcv(n=750)
frame.tail(3)

timestamp,symbol,open,high,low,close,volume,vwap,dollar_volume,market_ret
"datetime[μs, UTC]",str,f64,f64,f64,f64,f64,f64,f64,f64
2023-01-18 00:00:00 UTC,"""GRCH""",43.758513,43.944892,43.309043,43.661097,3.15592e6,43.638344,1.3779e8,0.010082
2023-01-19 00:00:00 UTC,"""GRCH""",42.897073,43.104998,42.716123,42.948482,3.773722e6,42.923201,1.6208e8,-0.005878
2023-01-20 00:00:00 UTC,"""GRCH""",44.463069,44.747985,44.163015,44.489422,2.506017e6,44.466807,1.1149e8,-0.004576


## Compute the estimators
Six volatility estimators on one window, plus ATR and the up/down vol ratio. The estimators are *return-std per bar*; we annualize by √252 for plotting (ATR is in price units, so we normalize it by close).

In [2]:
ESTIMATORS = {
    "vol_cc_21": sabia.volatility.vol_cc(window=21),          # close-to-close
    "vol_parkinson_21": sabia.volatility.vol_parkinson(window=21),
    "vol_gk_21": sabia.volatility.vol_gk(window=21),          # Garman-Klass
    "vol_rs_21": sabia.volatility.vol_rs(window=21),          # Rogers-Satchell
    "vol_yz_21": sabia.volatility.vol_yz(window=21),          # Yang-Zhang
    "vol_ewma_94": sabia.volatility.vol_ewma(lam=0.94),       # RiskMetrics EWMA
}

out = sabia.compute(
    frame,
    *ESTIMATORS.values(),
    sabia.volatility.atr(window=14),
    sabia.distribution.up_down_vol_ratio(window=63),
    schema=SCHEMA,
)
df = (
    frame.select("timestamp", "close")
    .hstack(out)
    .with_columns(
        (pl.col("close").log() - pl.col("close").log().shift(1)).alias("ret"),
        (pl.col("atr_14") / pl.col("close")).alias("atr_pct"),  # ATR is price units; normalize
    )
)
ret = df.get_column("ret").to_numpy()
ret = ret[~np.isnan(ret)]
t = df.get_column("timestamp").to_list()
print(f"{len(ret)} daily returns | realized vol {ret.std() * ANN:.1%} annualized")

749 daily returns | realized vol 22.1% annualized


## 1 · Clustering
The rolling `vol_cc(21)` traces out persistent calm and turbulent regimes — a random walk would not.

In [3]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.5, 0.5],
    subplot_titles=("Daily returns — calm and turbulent stretches alternate",
                    "Rolling close-to-close volatility, vol_cc(21), annualized %"),
)
fig.add_trace(go.Scatter(x=t, y=df.get_column("ret").to_list(), name="return",
                         line=dict(color="#455a64", width=0.7)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=(df.get_column("vol_cc_21") * ANN * 100).to_list(), name="vol_cc",
                         line=dict(color="#d62728", width=1.3), fill="tozeroy",
                         fillcolor="rgba(214,39,40,0.10)"), row=2, col=1)
fig.update_layout(height=520, showlegend=False,
                  title="Volatility clustering — large moves bunch together (GARCH simulation)",
                  margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## …quantified with the autocorrelation function
Returns sit inside the white-noise band; their **absolute value** decays slowly and stays positive — the signature of clustering.

In [4]:
nlags = 30
acf_r = st.acf(ret, nlags)[1:]
acf_abs = st.acf(np.abs(ret), nlags)[1:]
lags = list(range(1, nlags + 1))
conf = 1.96 / np.sqrt(len(ret))  # ~95% white-noise band

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "ACF of returns — essentially white noise", "ACF of |returns| — slow, positive decay"))
fig.add_trace(go.Bar(x=lags, y=acf_r, marker_color="#90a4ae", showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=lags, y=acf_abs, marker_color="#d62728", showlegend=False), row=1, col=2)
for c in (1, 2):
    fig.add_hrect(y0=-conf, y1=conf, row=1, col=c, fillcolor="#1f77b4", opacity=0.10, line_width=0)
fig.update_yaxes(range=[-0.15, 0.35])
fig.update_layout(height=380, title="Returns are unpredictable; their magnitude is not",
                  margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## 2 · Estimator efficiency
All six estimate the *same* underlying volatility. The close-to-close line (thick grey) is the jumpiest; the range/OHLC estimators are visibly smoother.

In [5]:
sl = df.tail(420)  # tail where every estimator (incl. the long-warmup EWMA) is non-null
ts = sl.get_column("timestamp").to_list()
palette = {"vol_cc_21": "#9e9e9e", "vol_parkinson_21": "#1f77b4", "vol_gk_21": "#2ca02c",
           "vol_rs_21": "#9467bd", "vol_yz_21": "#d62728", "vol_ewma_94": "#ff7f0e"}

fig = go.Figure()
for name, color in palette.items():
    width = 2.0 if name == "vol_cc_21" else 1.3
    fig.add_trace(go.Scatter(x=ts, y=(sl.get_column(name) * ANN * 100).to_list(),
                             name=name, line=dict(color=color, width=width)))
fig.add_trace(go.Scatter(x=ts, y=(sl.get_column("atr_pct") * ANN * 100).to_list(),
                         name="atr_14/close", line=dict(color="#17becf", width=1.0, dash="dot")))
fig.update_layout(height=460,
                  title="Same true volatility, different estimators — range/OHLC estimators track it more smoothly",
                  yaxis_title="annualized volatility (%)",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
                  margin=dict(l=60, r=30, t=90, b=40))
fig.show()

A simple efficiency proxy: how much each estimator jitters day-to-day. Lower bars = less sampling noise for the same signal.

In [6]:
# Efficiency proxy: a noisier estimator jumps around more day-to-day for the same underlying vol.
# Range/OHLC estimators (Parkinson, GK, RS, YZ) use intrabar information, so they are smoother than
# close-to-close (Yang-Zhang 2000).
names = list(palette.keys())
noise = {n: float(np.nanstd(np.diff((df.get_column(n) * ANN * 100).to_numpy()))) for n in names}
order = sorted(noise, key=noise.get)
fig = go.Figure(go.Bar(x=order, y=[noise[n] for n in order],
                       marker_color=[palette[n] for n in order],
                       text=[f"{noise[n]:.2f}" for n in order], textposition="outside"))
fig.update_layout(height=400,
                  title="Estimator sampling noise (std of day-over-day change) — lower is more efficient",
                  yaxis_title="vol points / day", margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## 3 · Leverage effect
`up_down_vol_ratio` compares upside to downside semi-deviation. Persistently below 1 means downside moves dominate the variance — the asymmetry our GJR leverage term injects.

In [7]:
# Leverage effect: downside shocks raise volatility more than upside shocks. up_down_vol_ratio < 1
# means downside semi-deviation dominates.
udr = df.get_column("up_down_vol_ratio_63").to_numpy()
med = float(np.nanmedian(udr))
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=udr.tolist(), name="up/down vol ratio",
                         line=dict(color="#6a3d9a", width=1.2)))
fig.add_hline(y=1.0, line=dict(color="#455a64", width=1, dash="dash"),
              annotation_text="symmetric (ratio = 1)")
fig.add_hline(y=med, line=dict(color="#d62728", width=1, dash="dot"),
              annotation_text=f"median {med:.2f}")
fig.update_layout(height=380,
                  title="Leverage effect — up_down_vol_ratio(63) sits below 1: downside vol runs hotter",
                  margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## Takeaways

- Volatility is **forecastable** even when direction is not — `vol_cc`, `vol_ewma`, and the range estimators all exploit this.
- Prefer **range/OHLC estimators** when you have OHLC bars: same signal, less noise.
- `up_down_vol_ratio` / `semivar_down` capture the **downside asymmetry** that symmetric vol misses — useful for risk-aware features.